# Dynamic Auto-Tagging with RAG-based Few-Shot Learning

This notebook demonstrates how to use dynamic prompting with TF-IDF retrieval for ticket classification.

## Setup & Initialization

In [7]:
import sys
import pandas as pd
import nest_asyncio

# 1. ALLOW ASYNC IN JUPYTER
nest_asyncio.apply()

# 2. FIX PATH TO PROJECT ROOT
sys.path.append('../../')

# Import the dynamic tagger
from utils.tagging.label_creation_dynamic_cisc import DynamicAutoTagger

print("Initializing Dynamic Tagger...")
# Initialize with config - using ../../ to reach project root
tagger = DynamicAutoTagger(
    config_path='../../config/config_labels_dynamic.yaml',
    # Silver Standard RAG database
    example_db_path='/afh/projects/SY-Tagging-NOT_DELETE-7b113334-385d-4e41-87b7-c77611f91817/shared/Users/josta/Thesis_IT_TicketClassification/data/labeled/lexicon_scenarios_1500.csv',  
    use_async=True 
)

print(f"✓ RAG Database loaded with {len(tagger.example_df)} examples.")
tagger.test_connection()

INFO:utils.tagging.label_creation_dynamic_cisc:Loading example database from /afh/projects/SY-Tagging-NOT_DELETE-7b113334-385d-4e41-87b7-c77611f91817/shared/Users/josta/Thesis_IT_TicketClassification/data/labeled/lexicon_scenarios_1500.csv
INFO:utils.tagging.label_creation_dynamic_cisc:Loaded 1331 labeled examples for RAG database.
INFO:utils.tagging.label_creation_dynamic_cisc:Initializing TF-IDF retriever...


Initializing Dynamic Tagger...


INFO:utils.tagging.label_creation_dynamic_cisc:✓ TF-IDF retriever initialized


✓ RAG Database loaded with 1331 examples.


INFO:httpx:HTTP Request: POST https://ai-coe-openai-models.openai.azure.com/openai/deployments/gpt-5/chat/completions?api-version=2025-01-01-preview "HTTP/1.1 200 OK"
INFO:utils.tagging.label_creation_dynamic_cisc:✓ Connection successful!


True

## Test the Retrieval System (Sanity Check)

In [8]:
# Test query
test_text = "Customer cannot login to YouSee Play app. Getting error message."
print(f"TEST QUERY: '{test_text}'\n")

# Retrieve similar examples
retrieved = tagger.retrieve_examples(test_text)

print(f"Retrieved {len(retrieved)} examples:")
print("-" * 50)
for i, ex in enumerate(retrieved, 1):
    print(f"Example {i} (Similarity: {ex['similarity']:.3f})")
    print(f"Text: {ex['text'][:120]}...")
    print(f"Label: {ex['label']}\n")

TEST QUERY: 'Customer cannot login to YouSee Play app. Getting error message.'

Retrieved 3 examples:
--------------------------------------------------
Example 1 (Similarity: 0.253)
Text: Support - Login/My YouSee/Other When the Customer logs into his My YouSee, only YouSee Play appears and not his mobile. ...
Label: (1g Self service, 2g Mit YouSee), 3 Missing rights to access

Example 2 (Similarity: 0.226)
Text: Support - TV/Other/Rights Customer cannot use their subscription when logging in with the correct login. It states that ...
Label: (1g Self service, 2g YS-play), 3 Missing rights to access

Example 3 (Similarity: 0.224)
Text: Support - TV/Video on demand/iOS App Customer wants to watch DR1 on her iPad using the YouSee Play app, but she keeps ge...
Label: (1g Self service, 2g YS-play), 3 technical issues



## Load the Target Data

In [10]:
# Load your tickets to classify
df = pd.read_csv('/afh/projects/SY-Tagging-NOT_DELETE-7b113334-385d-4e41-87b7-c77611f91817/shared/Users/josta/Thesis_IT_TicketClassification/data/labeled/distill_scenarios_25000.csv')

# Create text column
#df['text'] = df['title'] + ' ' + df['description'] + ' ' + df['assignment_group']

# Select columns for prediction
prediction_df = df[["number", "text"]].copy()

print(f"Loaded {len(prediction_df)} tickets to classify.")
prediction_df.head(3)

Loaded 25000 tickets to classify.


,number,text
0,INC0129809,Support - Dawn/With Customer context/Quotes an...
1,INC0122752,Support - Login/My YouSee/New Customer (first ...
2,INC0129065,PARENT ID - INC0118835 - Support - YouSee mail...


## Run the Dynamic CISC Pipeline

In [ ]:
# Run predictions with Dynamic Confidence-Informed Self-Consistency (CISC)
# - n_samples=10: Runs 10 independent reasoning paths per ticket
# - Output: A CSV with 'scientific_confidence', 'consistency', and dynamic 'shot' columns

test_df = prediction_df.head(10).copy()

result_df = tagger.predict_and_create_csv(
    df=test_df,
    output_path='../../data/dynamic_predictions_cisc_test.csv',
    number_column='number',
    text_column='text',
    n_samples=5,        # <--- Request 10 samples for the CISC voting
    semaphore_limit=10   # Lower this if you hit Rate Limits (10 samples * 10 tickets = 100 concurrent calls)
)

print(f"\nTotal predictions: {len(result_df)}")
print(f"Unique tickets: {result_df['number'].nunique()}")

# Verify the new columns
# You should see: 'scientific_confidence', 'consistency', and the retrieved 'shot1', 'shot2', etc.
result_df.head(10)

Processing (Dynamic CISC, N=10):   0%|                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                          | 0/10 [00:00<?, ?it/s]I


Total predictions: 10
Unique tickets: 10


,number,text,label,reasoning,scientific_confidence,reasoning_confidence,consistency,shot1,shot2,shot3
0,INC0122865,Support - TV/Video on demand/Streamer or Audio...,UNKNOWN,All samples failed parsing,0.0,0.0,0.0,Support - TV/Video on demand/SmartTV Creating ...,Support - TV/Other/Other Customer experiences ...,
1,INC0122830,Support - Dawn/With Customer context/Other I c...,UNKNOWN,All samples failed parsing,0.0,0.0,0.0,[Stuck pending cancelation]]Support - Dawn/Wit...,Support - Dawn/With Customer context/Products/...,Support - Dawn/With Customer context/Tickets/O...
2,INC0122818,Support - Dawn/With Customer context/Quotes an...,UNKNOWN,All samples failed parsing,0.0,0.0,0.0,[Stuck pending cancelation]]Support - Dawn/Wit...,Support - Dawn/With Customer context/Products/...,[FO seq: 72500] Support - Issues with ongoing ...
3,INC0122853,Support - TV/Live TV /Streamer or Audio Custom...,UNKNOWN,All samples failed parsing,0.0,0.0,0.0,Support - TV/Video on demand/SmartTV Creating ...,Support - TV/Other/Other Customer experiences ...,[Stuck pending cancelation]]Support - Dawn/Wit...
4,INC0122872,Support - Dawn/With customer context/Quotes an...,UNKNOWN,All samples failed parsing,0.0,0.0,0.0,[Stuck pending cancelation]]Support - Dawn/Wit...,[FO seq: 72500] Support - Issues with ongoing ...,Support - Dawn/With Customer context/Products/...
5,INC0122828,Support - Dawn/With Customer context/Quotes an...,UNKNOWN,All samples failed parsing,0.0,0.0,0.0,[Stuck pending cancelation]]Support - Dawn/Wit...,Support - Dawn/With Customer context/Products/...,[FO seq: 40600]Support - Dawn/Quotation/Cannot...
6,INC0122841,Support - Dawn/With Customer context/Products/...,UNKNOWN,All samples failed parsing,0.0,0.0,0.0,[FO seq: 40600]Support - Dawn/Quotation/Cannot...,[Stuck pending cancelation]]Support - Dawn/Wit...,Support - Dawn/With Customer context/Products/...
7,INC0122873,Support - TV/Other/Rights Customer is met with...,UNKNOWN,All samples failed parsing,0.0,0.0,0.0,Support - Login/My YouSee/Migrated Customer (b...,Support - TV/Other/Other Customer experiences ...,Support - TV/Video on demand/SmartTV Creating ...
8,INC0122831,Support - Dawn/Quotation/Andet Customer has mo...,UNKNOWN,All samples failed parsing,0.0,0.0,0.0,Support - Dawn/With Customer context/Products/...,[Stuck pending cancelation]]Support - Dawn/Wit...,[FO seq: 40600]Support - Dawn/Quotation/Cannot...
9,INC0122868,Support - TV/Video on demand/Streamer or Audio...,UNKNOWN,All samples failed parsing,0.0,0.0,0.0,Support - TV/Video on demand/SmartTV Creating ...,Support - TV/Other/Other Customer experiences ...,


In [16]:
# 1. Grab a sample ticket and its RAG examples
sample_text = prediction_df.iloc[23000]['text']
retrieved = tagger.retrieve_examples(sample_text)
examples_text = tagger._format_examples(retrieved)

# 2. Fill the template with real data to measure size
full_prompt_sample = tagger.prompt_template.format(
    hierarchical_categories=tagger.hierarchical_categories,
    scenarios=tagger.scenarios,
    examples_text=examples_text,
    description=sample_text
)

# 3. Calculate metrics
char_count = len(full_prompt_sample)
est_tokens = char_count / 4  # Standard industry heuristic for English/Danish mix

print(f"--- Token Estimation for 1 Sample ---")
print(f"Total Characters: {char_count:,}")
print(f"Estimated Tokens: {est_tokens:,.0f}")
print(f"Estimated Cost (Input): ${(est_tokens / 1_000_000) * 1.25:.4f} per sample")
print(f"Total Estimated Cost for 25,000 tickets (5 samples each): ${(est_tokens * 25000 * 5 / 1_000_000) * 1.25:,.2f}")

--- Token Estimation for 1 Sample ---
Total Characters: 4,453
Estimated Tokens: 1,113
Estimated Cost (Input): $0.0014 per sample
Total Estimated Cost for 25,000 tickets (5 samples each): $173.95


In [6]:
import asyncio

async def debug_llm_output():
    # 1. Use the same test ticket from your previous check
    test_text = prediction_df.iloc[0]['text']
    
    # 2. Get the RAG examples
    retrieved = tagger.retrieve_examples(test_text)
    examples_text = tagger._format_examples(retrieved)
    
    # 3. Request just ONE raw sample (no parsing yet)
    print("Requesting raw response from Azure OpenAI...")
    raw_responses = await tagger.predict_samples_async(
        description=test_text,
        examples_text=examples_text,
        n=1,
        temperature=1.0
    )
    
    print("\n--- RAW AI RESPONSE (THE 'TRUTH') ---")
    print(raw_responses[0])
    print("-------------------------------------")

# Run it in Jupyter
await debug_llm_output()

Requesting raw response from Azure OpenAI...


INFO:httpx:HTTP Request: POST https://ai-coe-openai-models.openai.azure.com/openai/deployments/gpt-5/chat/completions?api-version=2025-01-01-preview "HTTP/1.1 200 OK"



--- RAW AI RESPONSE (THE 'TRUTH') ---
(1c TV, 2c OTT), 3 technical issues

Reasoning: The problem concerns live TV functionality (rewind/start-over) on a YouSee Play Streamer device, which is part of the OTT TV service rather than a coax/fiber TV box. This is a service feature/technical issue on the OTT platform.

Confidence: 0.90
-------------------------------------


## Examine & Analyze Results

In [ ]:
# Load results
results_df = pd.read_csv('../../data/dynamic_predictions_cisc_test.csv')

print(f"Total rows predicted: {len(results_df)}")
print("-" * 50)

# Pick the first ticket to examine deeply
first_ticket = results_df.iloc[0]

print(f"TICKET: {first_ticket['number']}")
print(f"TEXT: {first_ticket['text'][:150]}...\n")

print(f"FINAL PREDICTION:")
print(f"  Label: {first_ticket['label']}")
print(f"  Scientific Confidence (S*): {first_ticket['scientific_confidence']}")
print(f"  Consistency (Voting Agreement): {first_ticket['consistency']}\n")

print(f"AI REASONING:")
print(f"  {first_ticket['reasoning']}\n")

print(f"DYNAMIC EXAMPLES USED (RAG Context):")
print(f"  Shot 1: {first_ticket.get('shot1', 'N/A')}")
print(f"  Shot 2: {first_ticket.get('shot2', 'N/A')}")
print(f"  Shot 3: {first_ticket.get('shot3', 'N/A')}")

## 8. Run on Full Dataset

In [ ]:
# Uncomment to run on full dataset
# results_full = tagger.predict_and_create_csv(
#     df=prediction_df,
#     output_path='../data/dynamic_predictions_full.csv',
#     number_column='number',
#     text_column='text'
# )

## 9. Compare Static vs Dynamic Prompting

In [ ]:
# Load static predictions (if you have them)
# static_results = pd.read_csv('../data/prediction_results.csv')
# dynamic_results = pd.read_csv('../data/dynamic_predictions_full.csv')

# # Compare top1 accuracy or other metrics
# # Add your comparison analysis here

## Key Differences from Static Prompting:

### Static Prompting:
- Uses same 7 hardcoded examples for ALL tickets
- Examples may not be relevant to the input

### Dynamic Prompting (This Notebook):
- Retrieves 3 most similar examples for EACH ticket
- Examples are always relevant to the input
- Stores which examples were used (shot1, shot2, shot3)
- Can analyze which examples lead to better predictions

### Output Format:
```
number | text | label | reasoning | confidence_score | shot1 | shot2 | shot3
```

Each ticket gets 3 rows (top3 predictions), all with the same retrieved examples.